
# Retrieving Complete Taxonomy Lineages from NCBI with Python (Biopython Entrez)

This notebook provides a robust, reusable workflow to map **partial taxonomy data** (e.g., names at a single rank)
to **complete hierarchical lineages** using the NCBI Taxonomy API via **Biopython's `Bio.Entrez`**.

## Highlights
- Uses `Entrez.esearch` + `Entrez.efetch` with **structured parsing** from `LineageEx` (rank–name pairs).
- Handles **ambiguous searches**, **multiple TaxIDs**, and **network/API errors** with retries.
- Respects **NCBI usage guidelines** (set email, optional API key, basic rate limiting).
- Batch processing with optional **caching** to CSV to avoid repeated API calls.
- Produces a **clean DataFrame** with standard bacterial ranks (Domain, Phylum, Class, Order, Family, Genus).



## Requirements

- Python 3.9+
- `biopython` and `pandas`

```
pip install biopython pandas
```


In [4]:

from __future__ import annotations

import os
import time
from typing import Dict, List, Optional

import pandas as pd
from Bio import Entrez
from urllib.error import HTTPError


In [ ]:

## Configuration

Set your contact email (required by NCBI) and optional API key.

- You can pass them as function arguments, or
- Set environment variables `NCBI_EMAIL` and `NCBI_API_KEY`.


In [5]:

def _configure_entrez(email: Optional[str] = None, api_key: Optional[str] = None) -> None:
    email = email or os.getenv("NCBI_EMAIL")
    if not email:
        raise ValueError("NCBI email is required. Pass `email=` or set NCBI_EMAIL env var.")
    Entrez.email = email
    Entrez.api_key = api_key or os.getenv("NCBI_API_KEY") or None



## Core: Fetch a Single Taxon's Lineage

The function below:
- Searches a **specific rank** (e.g., Genus) using `esearch`.
- Chooses the **best matching TaxID** when multiple results are returned.
- Fetches the record with `efetch` and parses **`LineageEx`** (rank–name pairs).
- Returns a dictionary containing standard bacterial ranks and metadata.


In [ ]:

STANDARD_RANKS = ["Domain", "Phylum", "Class", "Order", "Family", "Genus"]

def _choose_best_taxid(esearch_record: dict) -> Optional[str]:
    ids = esearch_record.get("IdList", [])
    if not ids:
        return None
    return ids[0]

def get_taxonomy_lineage(
    taxon_name: str,
    rank: str = "Genus",
    email: Optional[str] = None,
    api_key: Optional[str] = None,
    retries: int = 3,
    delay_sec: float = 0.34
) -> Optional[Dict[str, str]]:
    """Fetch a taxon's lineage from NCBI Taxonomy using LineageEx.

    Parameters
    ----------
    taxon_name : str
        Name of the taxon to search (e.g., "Lactobacillus").
    rank : str, default "Genus"
        Rank to search at (e.g., "Genus", "Family").
    email : Optional[str]
        NCBI contact email (required). If None, uses env var `NCBI_EMAIL`.
    api_key : Optional[str]
        NCBI API key to increase rate limit. If None, uses env var `NCBI_API_KEY`.
    retries : int, default 3
        Number of times to retry on errors (HTTPError, network hiccups).
    delay_sec : float, default 0.34
        Delay between requests to respect rate limits (≈3 req/s without API key).

    Returns
    -------
    dict or None
        Dictionary with keys:
          - TaxID, ScientificName, Rank, and standardized bacterial ranks
            (Domain, Phylum, Class, Order, Family, Genus) when available.
        Returns None if no record was found.
    """
    _configure_entrez(email=email, api_key=api_key)

    term = f"{taxon_name}[{rank}]"
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            time.sleep(delay_sec)
            with Entrez.esearch(db="taxonomy", term=term) as h:
                esearch_record = Entrez.read(h)
            taxid = _choose_best_taxid(esearch_record)
            if not taxid:
                return None

            time.sleep(delay_sec)
            with Entrez.efetch(db="taxonomy", id=taxid, retmode="xml") as h:
                records = Entrez.read(h)

            if not records:
                return None

            rec = records[0]
            scientific_name = rec.get("ScientificName", "")
            actual_rank = rec.get("Rank", "")
            lineage_ex = rec.get("LineageEx", [])

            lineage_map = {k: None for k in STANDARD_RANKS}

            rank_alias = {
                "superkingdom": "Domain",
                "phylum": "Phylum",
                "class": "Class",
                "order": "Order",
                "family": "Family",
                "genus": "Genus",
            }

            for node in lineage_ex:
                r = node.get("Rank", "").lower()
                name = node.get("ScientificName", None)
                if r in rank_alias and name:
                    lineage_map[rank_alias[r]] = name

            if actual_rank.lower() in rank_alias:
                lineage_map[rank_alias[actual_rank.lower()]] = scientific_name or taxon_name
            elif rank in STANDARD_RANKS:
                lineage_map[rank] = scientific_name or taxon_name

            result = {
                "TaxID": rec.get("TaxId", ""),
                "ScientificName": scientific_name or taxon_name,
                "Rank": actual_rank or rank,
            }
            result.update(lineage_map)
            return result

        except HTTPError as e:
            last_err = e
            if e.code in (400, 404):
                break
            if attempt == retries:
                raise
            time.sleep(delay_sec * attempt)
        except Exception as e:
            last_err = e
            if attempt == retries:
                raise
            time.sleep(delay_sec * attempt)

    return None



## Batch Processing with Optional Caching

The utility below processes a list of taxa (at a given rank) and optionally uses a CSV cache to
avoid repeated API calls across runs.


In [ ]:

def fetch_lineages_batch(
    taxa: List[str],
    rank: str = "Genus",
    email: Optional[str] = None,
    api_key: Optional[str] = None,
    cache_csv: Optional[str] = None,
    retries: int = 3,
    delay_sec: float = 0.34,
    verbose: bool = True
) -> pd.DataFrame:
    existing = pd.DataFrame()
    if cache_csv and Path(cache_csv).exists():
        existing = pd.read_csv(cache_csv, dtype=str)
        if 'ScientificName' in existing.columns:
            already = set(existing['ScientificName'].astype(str).str.strip())
        else:
            already = set()
    else:
        already = set()

    remaining = [t for t in taxa if t not in already]

    rows = []
    if verbose:
        print(f"Total taxa: {len(taxa)} | Already cached: {len(already)} | To fetch: {len(remaining)}")

    for i, name in enumerate(remaining, 1):
        if verbose:
            print(f"[{i}/{len(remaining)}] Fetching: {name}")
        rec = get_taxonomy_lineage(
            taxon_name=name,
            rank=rank,
            email=email,
            api_key=api_key,
            retries=retries,
            delay_sec=delay_sec
        )
        if rec:
            rows.append(rec)
        else:
            rows.append({
                "TaxID": None,
                "ScientificName": name,
                "Rank": None,
                **{k: None for k in STANDARD_RANKS}
            })

    new_df = pd.DataFrame(rows, columns=["ScientificName", "TaxID", "Rank", *STANDARD_RANKS])
    new_df = new_df[["ScientificName", "TaxID", "Rank", *STANDARD_RANKS]]

    if not existing.empty:
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined = combined.sort_values(by=["ScientificName"]).drop_duplicates(subset=["ScientificName"], keep="last")
    else:
        combined = new_df

    if cache_csv:
        combined.to_csv(cache_csv, index=False)

    return combined



## Example Usage (Genus-Level)

**Note:** Running the cells below will make live requests to NCBI.  
Set your email via `NCBI_EMAIL` env var or pass `email=` directly.


In [ ]:

genera = [
    'Acetobacter', 'Gluconacetobacter', 'Lentibacillus', 'Brevibacterium',
    'Kosakonia', 'Lactobacillus', 'Companilactobacillus', 'Schleiferilactobacillus',
    'Lactiplantibacillus', 'Loigolactobacillus', 'Paucilactobacillus', 'Limosilactobacillus',
    'Acetilactobacillus', 'Secundilactobacillus', 'Lentilactobacillus', 'Carnobacterium',
    'Enterococcus', 'Tetragenococcus', 'Streptococcus', 'Lactococcus', 'Pediococcus',
    'Marinilactobacillus', 'Alkalibacterium', 'Eggerthella', 'Propionibacterium',
    'Weissella', 'Oenococcus', 'Leuconostoc', 'Periweissella', 'Staphylococcus', 'Kocuria'
]

# Example (commented): replace with your email or set NCBI_EMAIL
# df = fetch_lineages_batch(genera, rank="Genus", email="your_email@example.com", cache_csv="lineages_cache.csv")
# df.head()



## Notes & Best Practices

- **Email is required** by NCBI (`Entrez.email`) to help them contact you in case of issues.
- **API key** (`Entrez.api_key`) is optional but recommended for higher throughput (10 req/s limit).
- Keep a **delay** between requests (e.g., ~0.34s without API key ≈ 3 req/s); increase if you see errors.
- Prefer parsing **`LineageEx`** for structured rank–name pairs rather than splitting the lineage string.
- Use **caching** when working with large lists to minimize repeat calls and stay within rate limits.
- Lineage content and naming can change as NCBI updates its taxonomy; consider persisting snapshots.
